# CatBoostClassifier: четыре класса задержки
Классы по `target_delay_s`: `<−60`, `[−60,60)`, `[60,300)`, `≥300` секунд. Число деревьев выбирается на трёх отложенных группах исходных ТС. Test используется только для итогового отчёта; результаты сохраняются на Drive.


In [ ]:
DRIVE_ZIP = '/content/drive/MyDrive/Mos-TRANS/dataset.zip'
DRIVE_ROOT = '/content/drive/MyDrive/Mos-TRANS/catboost-classifier'
REPO_URL = 'https://github.com/epitaph76/Mos-TRANS.git'
BRANCH = 'ya-dolbayob'


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
assert Path(DRIVE_ZIP).is_file(), f'Архив не найден: {DRIVE_ZIP}'


In [ ]:
import subprocess, sys, shutil, hashlib
repo = Path('/content/Mos-TRANS')
if not repo.exists():
    subprocess.run(['git', 'clone', '--single-branch', '--branch', BRANCH, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements-catboost.txt')], check=True)
if str(repo) not in sys.path: sys.path.insert(0, str(repo))


In [ ]:
from mos_trans.preprocessing import build_dataset
from mos_trans.modeling.classifier import run, Config
work = Path('/content/mos-trans-classifier')
work.mkdir(parents=True, exist_ok=True)
local_zip = work / 'dataset.zip'
shutil.copy2(DRIVE_ZIP, local_zip)
with local_zip.open('rb') as source:
    dataset_hash = hashlib.file_digest(source, 'sha256').hexdigest()
processed = work / 'processed'
if not (processed / 'train_features.parquet').is_file():
    build_dataset(local_zip, processed)
output = Path(DRIVE_ROOT) / dataset_hash[:20]
report = run(processed, local_zip, output, Config())
print('Артефакты:', output)


In [ ]:
import pandas as pd
from IPython.display import display
display(pd.DataFrame(report['holdout'])[['seed','held_out_vehicles','trees','macro_f1','balanced_accuracy','log_loss']])
print('Средний holdout macro F1:', round(report['holdout_macro_f1_mean'], 3))
print('Test:', {key: round(report['test'][key], 3) for key in ('macro_f1','balanced_accuracy','log_loss')})
print('Файлы:', *[p.name for p in output.iterdir()])
